In [ ]:
!pip install openai PyPDF2 faiss-cpu numpy

In [ ]:
import os
import openai
import PyPDF2
import numpy as np
import faiss
import pickle
import json
import re
from google.colab import files, drive
from typing import List, Dict, Tuple, Set
from IPython.display import clear_output, display, HTML
import time
import warnings
warnings.filterwarnings("ignore")

# Interactive chat interface for Colab
from IPython.display import HTML, display

class HealthcareTutorSystem:
    """
    A healthcare tutoring system that implements the Ruffle-Riley framework
    using RAG, TASBox, and dialogue management techniques.
    """

    def __init__(self):
        """Initialize the healthcare tutor system."""
        # Mount Google Drive for persistence
        drive.mount('/content/drive')
        self.DRIVE_PATH = '/content/drive/MyDrive/healthcare_chatbot/'

        # Create directory if it doesn't exist
        if not os.path.exists(self.DRIVE_PATH):
            os.makedirs(self.DRIVE_PATH)

        # Initialize OpenAI client
        self.setup_openai()

        # Load or process PDF
        self.pdf_file = self.upload_pdf()

        # Initialize system components
        self.book_text = ""
        self.chunks = []
        self.embeddings = []
        self.index = None
        self.scripts = []

        # Initialize tutoring session state
        self.current_script_idx = 0
        self.answered_expectations = set()
        self.current_script = None
        self.student_user_history = []
        self.misconception_detected = False

        # Process the uploaded PDF
        self.process_pdf()

    def setup_openai(self):
        """Set up the OpenAI API client with user-provided key."""
        api_key = input("Enter your OpenAI API key: ")
        os.environ["OPENAI_API_KEY"] = api_key
        self.client = openai.OpenAI(api_key=api_key)

    def upload_pdf(self) -> str:
        """Upload healthcare PDF book and return its filename."""
        print("Please upload your healthcare book (PDF):")
        uploaded = files.upload()
        return list(uploaded.keys())[0]

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extract text content from a PDF file."""
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = ""
            for page in reader.pages:
                text += page.extract_text() + "\n"
        return text

    def chunk_text(self, text: str, chunk_size: int = 500) -> List[str]:
        """Split text into manageable chunks for processing."""
        words = text.split()
        chunks = []
        for i in range(0, len(words), chunk_size):
            chunks.append(" ".join(words[i:i + chunk_size]))
        return chunks

    def get_embedding(self, text: str) -> List[float]:
        """Generate embeddings for text chunks using OpenAI's embedding model."""
        try:
            response = self.client.embeddings.create(
                model="text-embedding-ada-002",
                input=text
            )
            return response.data[0].embedding
        except Exception as e:
            print(f"Error generating embedding: {e}")
            return []

    def build_faiss_index(self, embeddings: List[List[float]]) -> faiss.IndexFlatL2:
        """Build a FAISS index for efficient similarity search."""
        dimension = len(embeddings[0])
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(embeddings).astype('float32'))
        return index

    def generate_tutoring_script(self, chunk: str) -> Dict:
        """
        Generate a tutoring script from a text chunk (TASBox component).
        This creates review questions, solutions, and learning expectations.
        """
        prompt = f"""
        Given the following healthcare text:
        {chunk}

        Generate a tutoring script with:
        1. One review question about an important concept.
        2. A comprehensive solution to the question.
        3. Three key learning expectations for the answer (these are the key points a student should cover).

        Return the result in JSON format with fields: question, solution, expectations (as a list of strings).
        """
        try:
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=500
            )
            content = response.choices[0].message.content.strip()
            # Clean and parse JSON
            json_str = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', content)
            if json_str.startswith('```json') and json_str.endswith('```'):
                json_str = re.search(r'```json\n(.*?)\n```', json_str, re.DOTALL).group(1).strip()
            script = json.loads(json_str)
            return script
        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
            return {"question": "", "solution": "", "expectations": []}
        except Exception as e:
            print(f"Error generating tutoring script: {e}")
            return {"question": "", "solution": "", "expectations": []}

    def retrieve_relevant_chunks(self, query: str, k: int = 3) -> List[str]:
        """Retrieve relevant content chunks for the given query (RAG component)."""
        query_embedding = self.get_embedding(query)
        if not query_embedding:
            return []
        distances, indices = self.index.search(np.array([query_embedding]).astype('float32'), k)
        return [self.chunks[i] for i in indices[0] if i < len(self.chunks)]

    def ruffle_ask_question(self) -> str:
        """
        Ruffle agent component: Ask questions and guide the learning process.

        Returns a prompt from Ruffle based on current learning progress.
        """
        if not self.current_script:
            return "Ruffle: Sorry, no question available. Let's move to the next topic."

        question = self.current_script.get('question', '')
        expectations = self.current_script.get('expectations', [])

        if not question or not expectations:
            return "Ruffle: Sorry, no question available. Let's move to the next topic."

        remaining = [e for i, e in enumerate(expectations) if i not in self.answered_expectations]

        if remaining:
            return f"Ruffle: {question}\nPlease explain, covering this key point: {remaining[0]}"
        else:
            return f"Ruffle: Great! You've covered all key points for this question. Let's move to the next one."

    def riley_provide_feedback(self, user_answer: str) -> Tuple[str, int, str]:
        """
        Riley agent component: Evaluate answers and provide feedback.

        Returns:
        - Feedback text
        - Index of covered expectation (-1 if none)
        - Misconception text (empty if none)
        """
        expectations = self.current_script.get('expectations', [])
        relevant_chunks = self.retrieve_relevant_chunks(user_answer)

        prompt = f"""
        Given the student's answer: {user_answer}

        Relevant book content: {relevant_chunks}

        Learning expectations: {expectations}

        Evaluate the answer and provide feedback:
        1. If it covers one of the expectations, identify which expectation number (0, 1, or 2) it covers
        2. If it contains a misconception, identify and explain it
        3. Provide helpful guidance to improve the answer

        Return a JSON with:
        - feedback: String feedback for the student (supportive and specific)
        - covered_expectation: Index of covered expectation (0, 1, 2, or -1 if none)
        - misconception: String describing misconception (or empty if none)
        """
        try:
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=400
            )
            content = response.choices[0].message.content.strip()

            # Clean and parse JSON
            json_str = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', content)
            if json_str.startswith('```json') and json_str.endswith('```'):
                json_str = re.search(r'```json\n(.*?)\n```', json_str, re.DOTALL).group(1).strip()

            result = json.loads(json_str)
            feedback = result.get('feedback', '')
            covered_expectation = result.get('covered_expectation', -1)
            misconception = result.get('misconception', '')

            return feedback, covered_expectation, misconception

        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
            return "I had trouble processing your answer. Could you please rephrase it?", -1, ''
        except Exception as e:
            print(f"Error in Riley feedback: {e}")
            return "I had trouble evaluating your answer. Let's try again.", -1, ''

    def riley_provide_help(self) -> str:
        """
        Riley agent component: Provide targeted help based on the current topic.

        Returns supportive guidance to help the student understand key concepts.
        """
        if not self.current_script:
            return "I don't have enough context to provide help. Let's start with a question first."

        prompt = f"""
        The student needs help with this question: {self.current_script.get('question', '')}

        The expected learning points are: {self.current_script.get('expectations', [])}

        Reference solution: {self.current_script.get('solution', '')}

        Provide a helpful hint that guides the student toward understanding without giving away the full answer.
        Be encouraging and supportive. Focus on conceptual understanding rather than just facts.
        """

        try:
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=250
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Error in Riley help response: {e}")
            return "I'm having trouble generating a hint right now. Let's try approaching the question step by step."

    def choose_topic(self):
        """Allow user to focus on a specific topic from the existing PDF."""
        print("What topic would you like to study from the current material?")
        topic = input("Enter topic (e.g., 'diabetes', 'patient care', etc.): ")

        if not topic.strip():
            self.display_message("No topic entered. Continuing with general questions.", "alert")
            return False

        self.display_message(f"Focusing on '{topic}'. Generating relevant questions...", "riley")

        # Create topic-specific questions using the existing PDF content
        topic_scripts = []
        relevant_chunks = self.retrieve_relevant_chunks(topic, k=5)  # Get chunks relevant to the topic

        if not relevant_chunks:
            self.display_message(f"Could not find content about '{topic}' in the current material.", "alert")
            return False

        print(f"Found {len(relevant_chunks)} relevant sections on '{topic}'.")
        print("Generating specialized questions...")

        # Generate scripts for each relevant chunk
        for i, chunk in enumerate(relevant_chunks):
            try:
                # Add topic to prompt for more focused questions
                prompt = f"""
                Given the following healthcare text about {topic}:
                {chunk}

                Generate a tutoring script with:
                1. One review question about an important concept related to {topic}.
                2. A comprehensive solution to the question.
                3. Three key learning expectations for the answer (these are the key points a student should cover).

                Format your response exactly like this:
                QUESTION: Your question here about {topic}
                SOLUTION: Your comprehensive solution here
                EXPECTATION1: First key point
                EXPECTATION2: Second key point
                EXPECTATION3: Third key point

                Do not include any other text or explanations besides these five elements.
                """

                response = self.client.chat.completions.create(
                    model="gpt-4",
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=600
                )
                content = response.choices[0].message.content.strip()

                # Parse the response into our expected format
                try:
                    # Extract components using regex
                    question_match = re.search(r"QUESTION:\s*(.*?)(?=SOLUTION:|$)", content, re.DOTALL)
                    solution_match = re.search(r"SOLUTION:\s*(.*?)(?=EXPECTATION1:|$)", content, re.DOTALL)
                    exp1_match = re.search(r"EXPECTATION1:\s*(.*?)(?=EXPECTATION2:|$)", content, re.DOTALL)
                    exp2_match = re.search(r"EXPECTATION2:\s*(.*?)(?=EXPECTATION3:|$)", content, re.DOTALL)
                    exp3_match = re.search(r"EXPECTATION3:\s*(.*?)(?=$)", content, re.DOTALL)

                    # Get the matches or default values
                    question = question_match.group(1).strip() if question_match else f"Question about {topic}"
                    solution = solution_match.group(1).strip() if solution_match else "Comprehensive answer about the topic"

                    # Collect expectations
                    expectations = []
                    if exp1_match:
                        expectations.append(exp1_match.group(1).strip())
                    if exp2_match:
                        expectations.append(exp2_match.group(1).strip())
                    if exp3_match:
                        expectations.append(exp3_match.group(1).strip())

                    # If missing expectations, add generic ones
                    while len(expectations) < 3:
                        expectations.append(f"Explain a key aspect of {topic}")

                    # Create script dictionary
                    script = {
                        "question": question,
                        "solution": solution,
                        "expectations": expectations
                    }

                    if script.get('question') and script.get('expectations'):
                        topic_scripts.append(script)
                        print(f"Generated question {len(topic_scripts)}: {question[:50]}...")
                except Exception as e:
                    print(f"Error parsing response for chunk {i+1}: {e}")

                # Small delay to avoid rate limiting
                time.sleep(1)

            except Exception as e:
                print(f"Error generating topic script: {e}")

        # If we found relevant scripts, use them
        if topic_scripts:
            self.scripts = topic_scripts
            self.current_script_idx = 0
            self.current_script = self.scripts[0]
            self.answered_expectations = set()
            self.display_message(f"Generated {len(topic_scripts)} questions about '{topic}'.", "riley")
            return True
        else:
            self.display_message(f"Couldn't generate questions about '{topic}'. Continuing with existing questions.", "alert")
            return False

    def process_pdf(self):
        """Process the PDF and prepare the tutoring system components."""
        print("Extracting text from PDF...")
        self.book_text = self.extract_text_from_pdf(self.pdf_file)
        self.chunks = self.chunk_text(self.book_text)

        # Load or generate embeddings and FAISS index
        embeddings_file = self.DRIVE_PATH + 'embeddings.pkl'
        index_file = self.DRIVE_PATH + 'faiss_index.pkl'

        if os.path.exists(embeddings_file) and os.path.exists(index_file):
            print("Loading existing embeddings and index...")
            with open(embeddings_file, 'rb') as f:
                self.embeddings = pickle.load(f)
            with open(index_file, 'rb') as f:
                self.index = pickle.load(f)
        else:
            print("Generating embeddings (this may take a while)...")
            self.embeddings = [self.get_embedding(chunk) for chunk in self.chunks]
            self.embeddings = [emb for emb in self.embeddings if emb]  # Filter out empty embeddings

            if not self.embeddings:
                raise ValueError("No embeddings generated. Check your API key and connection.")

            print("Building search index...")
            self.index = self.build_faiss_index(self.embeddings)

            # Save for future use
            with open(embeddings_file, 'wb') as f:
                pickle.dump(self.embeddings, f)
            with open(index_file, 'wb') as f:
                pickle.dump(self.index, f)

        # Load or generate tutoring scripts
        scripts_file = self.DRIVE_PATH + 'tutoring_scripts.json'

        if os.path.exists(scripts_file):
            print("Loading existing tutoring scripts...")
            with open(scripts_file, 'r') as f:
                self.scripts = json.load(f)
        else:
            print("Generating tutoring scripts (this may take a while)...")
            self.scripts = []

            # Process chunks in batches to avoid API rate limits
            batch_size = 5
            for i in range(0, len(self.chunks), batch_size):
                batch = self.chunks[i:i+batch_size]
                print(f"Processing batch {i//batch_size + 1}/{(len(self.chunks) + batch_size - 1)//batch_size}...")

                for chunk in batch:
                    script = self.generate_tutoring_script(chunk)
                    if script.get('question') and script.get('expectations'):
                        self.scripts.append(script)

                # Save progress after each batch
                with open(scripts_file, 'w') as f:
                    json.dump(self.scripts, f)

                # Small delay to avoid rate limiting
                time.sleep(1)

        # Filter out invalid scripts
        self.scripts = [script for script in self.scripts if script.get('question') and script.get('expectations')]
        print(f"Loaded {len(self.scripts)} tutoring scripts.")

        if not self.scripts:
            raise ValueError("No valid tutoring scripts available. Check the PDF content.")

    def start_session(self):
        """Start the tutoring session with interactive UI."""
        # Create UI for the tutoring session
        display(HTML("""
        <style>
        .chatbox {
            max-width: 800px;
            margin: 0 auto;
            padding: 10px;
            background-color: #f9f9f9;
            border-radius: 8px;
            margin-bottom: 10px;
        }
        .ruffle {
            background-color: #e1f5fe;
            padding: 10px;
            border-radius: 5px;
            margin-bottom: 5px;
        }
        .riley {
            background-color: #e8f5e9;
            padding: 10px;
            border-radius: 5px;
            margin-bottom: 5px;
        }
        .alert {
            background-color: #ffebee;
            padding: 10px;
            border-radius: 5px;
            margin-bottom: 5px;
        }
        </style>
        <div class="chatbox" id="chatbox">
            <h2>Healthcare Tutoring Session</h2>
            <p>The tutoring system is ready! Answer questions to test your knowledge of the healthcare material.</p>
        </div>
        """))

        # Start the first script
        if self.scripts:
            self.current_script = self.scripts[0]
            self.answered_expectations = set()
            self.run_tutoring_loop()

    def display_message(self, message, agent_type="ruffle"):
        """Display a message in the chat interface with appropriate styling."""
        if agent_type == "ruffle":
            display(HTML(f'<div class="ruffle"><strong>Ruffle:</strong> {message}</div>'))
        elif agent_type == "riley":
            display(HTML(f'<div class="riley"><strong>Riley:</strong> {message}</div>'))
        elif agent_type == "alert":
            display(HTML(f'<div class="alert"><strong>Alert:</strong> {message}</div>'))

    def run_tutoring_loop(self):
        """Run the main tutoring interaction loop with topic selection."""
        try:
            display(HTML("""
            <div style="background-color: #e8f5e9; padding: 10px; border-radius: 5px; margin: 10px 0;">
                <p><strong>Instructions:</strong> Answer the questions to demonstrate your knowledge.
                Type 'help' if you need a hint, 'skip' to move to the next question, 'exit' to end the session,
                or 'topic' to select a specific topic to study.</p>
            </div>
            """))

            while True:
                # Check if we've gone through all questions
                if self.current_script_idx >= len(self.scripts):
                    self.display_message("Congratulations! You've completed all the questions in this session.", "riley")
                    restart = input("\nWould you like to restart (yes/no) or study a specific topic (topic)? ")
                    if restart.lower() == "yes":
                        self.current_script_idx = 0
                        self.display_message("Restarting the session with the same material.", "riley")
                    elif restart.lower() == "topic":
                        self.choose_topic()
                    else:
                        self.display_message("Ending the tutoring session. Thank you for participating!", "riley")
                        break

                # Get the current script
                self.current_script = self.scripts[self.current_script_idx]
                self.answered_expectations = set()

                # Show progress
                display(HTML(f"""
                <div style="background-color: #e0f7fa; padding: 5px; border-radius: 5px; margin: 5px 0; text-align: center;">
                    <p>Question {self.current_script_idx + 1} of {len(self.scripts)}</p>
                </div>
                """))

                # Continue with current script until all expectations are covered
                expectation_attempts = 0
                max_attempts_per_question = 10

                while len(self.answered_expectations) < len(self.current_script['expectations']):
                    # Check if we've exceeded max attempts for this question
                    if expectation_attempts >= max_attempts_per_question:
                        self.display_message("Let's move on to the next question since we've spent enough time on this one.", "riley")
                        break

                    # Ruffle asks about the next expectation
                    ruffle_message = self.ruffle_ask_question()
                    self.display_message(ruffle_message, "ruffle")

                    # Get user input with error handling
                    try:
                        user_answer = input("\nYour answer (or type 'help', 'skip', 'topic', or 'exit'): ")
                    except Exception as e:
                        print(f"Input error: {e}")
                        user_answer = ""

                    # Handle special commands
                    if user_answer.lower() == "exit":
                        self.display_message("Ending the tutoring session. Thank you for participating!", "riley")
                        return
                    elif user_answer.lower() == "topic":
                        if self.choose_topic():
                            # Reset to first question of new topic
                            self.current_script_idx = 0
                            break
                        else:
                            continue
                    elif user_answer.lower() == "skip":
                        self.display_message("Skipping to the next question.", "riley")
                        break
                    elif user_answer.lower() == "help":
                        help_text = self.riley_provide_help()
                        self.display_message(help_text, "riley")
                        expectation_attempts += 1
                        continue

                    # Regular answer processing logic
                    self.student_user_history.append(user_answer)

                    # Riley evaluates the answer
                    feedback, covered_expectation, misconception = self.riley_provide_feedback(user_answer)

                    # Update the answered expectations
                    if covered_expectation >= 0 and covered_expectation < len(self.current_script['expectations']):
                        self.answered_expectations.add(covered_expectation)
                        self.display_message(f"✓ Good job covering: {self.current_script['expectations'][covered_expectation]}", "riley")

                    # Provide feedback
                    self.display_message(feedback, "riley")

                    # Handle misconceptions if detected
                    if misconception:
                        self.misconception_detected = True
                        self.display_message(f"Clarification: {misconception}", "riley")

                    expectation_attempts += 1

                # Move to the next script
                self.current_script_idx += 1

        except Exception as e:
            self.display_message(f"An error occurred: {str(e)}. Please restart the session.", "alert")

# Initialize and run the tutoring system
tutor = HealthcareTutorSystem()
tutor.start_session()